# Phase 2: Synthetic Control Method — Lahaina Wildfire

Three estimators: ADH (2010), GSynth/IFE (Xu 2017), ASCM (Ben-Michael et al. 2021)

In [ ]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.scm.model_registry import ModelRegistry
from src.outputs.scm_plots import plot_scm_path, plot_placebo_distribution, plot_loo, plot_model_comparison
from src.outputs.scm_tables import weights_table_latex, rmspe_table_latex, balance_table_latex

FIRE_DATE = '2023-08'
RESULTS_DIR = '../results/scm'
INFERENCE_DIR = '../results/inference'

In [ ]:
# 1. Load all SCM results
with open(f'{RESULTS_DIR}/adh_results.pkl', 'rb') as f:
    adh = pickle.load(f)
with open(f'{RESULTS_DIR}/gsynth_results.pkl', 'rb') as f:
    gsynth = pickle.load(f)
with open(f'{RESULTS_DIR}/augsynth_results.pkl', 'rb') as f:
    augsynth = pickle.load(f)

reg = ModelRegistry()
reg.register('ADH', adh, {})
reg.register('GSynth', gsynth, {})
reg.register('ASCM', augsynth, {})

adh_gap = pd.read_parquet(f'{RESULTS_DIR}/adh_gap_series.parquet')
zip_panel = pd.read_parquet('../data/interim/zip_panel.parquet')
time_periods = sorted(zip_panel['year_month'].unique())
print('Loaded results. Time periods:', len(time_periods))

In [ ]:
# 2. SCM path plot: Lahaina vs. synthetic
data = np.load('../data/interim/covariate_matrix.npz', allow_pickle=True)
Y0_pre, Y1_pre = data['Y0_pre'], data['Y1_pre']

Y_synth = adh.predict(Y0_pre)
plot_scm_path(Y1_pre, Y_synth, time_periods[:len(Y1_pre)], FIRE_DATE, '../figures/scm_path.pdf')
plt.show()

In [ ]:
# 3. Placebo distribution
placebo_df = pd.read_parquet(f'{INFERENCE_DIR}/placebo_distribution.parquet')
gap_treated = adh_gap['gap'].values
plot_placebo_distribution(placebo_df, gap_treated, time_periods[:len(gap_treated)], FIRE_DATE,
                          '../figures/placebo_distribution.pdf')
plt.show()

In [ ]:
# 4. Balance table
cov_data = np.load('../data/interim/covariate_matrix.npz', allow_pickle=True)
X0, X1 = cov_data['X0'], cov_data['X1']
cov_names = list(cov_data['covariate_names'])

donor_pool = pd.read_parquet('../data/interim/donor_pool.parquet')
donor_names = [z for z in donor_pool['zip_code'].unique() if z != '96761']

latex = balance_table_latex(X0, X1, donor_names, cov_names, adh.w_)
print(latex)

In [ ]:
# 5. Model comparison plot
plot_model_comparison(reg, time_periods[:len(gap_treated)], FIRE_DATE, '../figures/model_comparison.pdf')
plt.show()

In [ ]:
# 6. P-values summary
import json
with open(f'{INFERENCE_DIR}/p_values.json') as f:
    pvals = json.load(f)
with open(f'{INFERENCE_DIR}/stability_score.json') as f:
    stability = json.load(f)

print(f"RMSPE ratio (ADH): {pvals['rmspe_ratio']:.3f}")
print(f"In-space p-value:  {pvals['p_value']:.3f}")
print(f"LOO stability:     {stability['stability_score']:.4f}")
reg.compare_rmspe()